# Week 52, Day 1: Review I: membranes, neurons and synapses

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w052/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Problem 1: from concentrations to the resting potential

Write nernst(out, inside, z=1) in mV with RT/F = 26.7 mV, and ghk(pK, pNa, pCl) using the concentrations given (chloride enters with in and out swapped because its charge is negative). Then find, with brentq, the Na:K permeability ratio that gives -70 mV with no chloride permeability.

In [ ]:
import numpy as np
from scipy.optimize import brentq
RT_F = 26.7                                     # mV at 37 degrees C
conc = {"K": (5, 140), "Na": (145, 15), "Cl": (110, 10)}   # (outside, inside) in mM

def nernst(out, inside, z=1):
    return ...

def ghk(pK, pNa, pCl):
    num = pK * conc["K"][0] + pNa * conc["Na"][0] + pCl * conc["Cl"][1]
    den = ...
    return RT_F * np.log(num / den)

E = {ion: nernst(o, i, -1 if ion == "Cl" else 1) for ion, (o, i) in conc.items()}
V_rest = ghk(1, 0.05, 0.45)
ratio = brentq(lambda r: ghk(1, r, 0) + 70, 1e-4, 1)
print({ion: f"{e:.1f} mV" for ion, e in E.items()})
print(f"resting potential with pK:pNa:pCl = 1:0.05:0.45: {V_rest:.1f} mV")
print(f"Na:K permeability ratio for -70 mV without chloride: {ratio:.3f}")

In [ ]:
import numpy as np
def test_nernst():
    "nernst gives E_K near -89 mV and E_Na near +61 mV"
    assert abs(E["K"] + 89) < 1 and abs(E["Na"] - 60.6) < 1
def test_ghk():
    "ghk with the given permeabilities gives a resting potential between -75 and -60 mV"
    assert -75 < V_rest < -60
def test_ratio():
    "ratio is the Na:K permeability ratio (no chloride) that gives exactly -70 mV"
    assert abs(ghk(1, ratio, 0) + 70) < 1e-6

test_nernst()
test_ghk()
test_ratio()
print("All checks passed")

### Exercise: Problem 2: how adaptation reshapes the f-I curve

Write lif_rate_analytic(I) from the period formula (in spikes per second, with tau in ms), and complete simulate(I, b): Euler steps of tau dV/dt = -(V - EL) + R(I - w) and tau_w dw/dt = -w, with w increased by b at each spike. It returns the steady rate over the last half second and the first and last interspike intervals.

In [ ]:
tau, EL, Vth, Vr, R, tau_w, dt = 20.0, -70.0, -50.0, -65.0, 10.0, 200.0, 0.1   # ms, mV, MOhm, nA

def lif_rate_analytic(I):
    Vinf = EL + R * I
    if Vinf <= Vth:
        return 0.0
    return ...

def simulate(I, b, T=1000.0):
    V, w, spikes = EL, 0.0, []
    for k in range(int(T / dt)):
        V += dt / tau * (...)
        w += -dt / tau_w * w
        if V >= Vth:
            V = Vr; w += b; spikes.append(k * dt)
    s = np.array(spikes); isi = np.diff(s)
    rate = np.sum(s > T / 2) / (T / 2000)
    return rate, (isi[0] if len(isi) else np.inf), (isi[-1] if len(isi) else np.inf)

currents = np.array([1.5, 2.5, 3.0, 4.0, 6.0])
for I in currents:
    print(f"I = {I} nA: formula {lif_rate_analytic(I):5.1f} Hz, simulated {simulate(I, 0)[0]:5.1f} Hz, with adaptation {simulate(I, 0.5)[0]:5.1f} Hz")
r, first, last = simulate(4.0, 0.5)
print(f"with adaptation at 4 nA: first interval {first:.1f} ms, last {last:.1f} ms")

In [ ]:
import numpy as np
def test_analytic():
    "lif_rate_analytic is zero below rheobase and matches the period formula above it"
    assert lif_rate_analytic(1.5) == 0 and abs(lif_rate_analytic(3.0) - 1000 / (20 * np.log(25 / 10))) < 1e-9
def test_sim():
    "without adaptation the simulated rate matches the formula within 3%"
    assert abs(simulate(3.0, 0)[0] - lif_rate_analytic(3.0)) / lif_rate_analytic(3.0) < 0.03
def test_adapt():
    "adaptation lowers the steady rate at every current and the first interval is shorter than the last"
    r0 = np.array([simulate(I, 0)[0] for I in currents]); r1 = np.array([simulate(I, 0.5)[0] for I in currents])
    assert np.all(r1[r0 > 0] < r0[r0 > 0]) and simulate(4.0, 0.5)[1] < simulate(4.0, 0.5)[2]

test_analytic()
test_sim()
test_adapt()
print("All checks passed")

### Exercise: Problem 3: a depressing synapse as a rate filter

In the Tsodyks-Markram model each spike releases a fraction U of the available resources x, which recover towards 1 with time constant tau_rec. Write x_ss(rate), the available resources just before each spike in a regular train at steady state, simulate a train to check it, and find the rate at which the steady amplitude falls to half of the first.

In [ ]:
from scipy.optimize import brentq
U, tau_rec = 0.5, 800.0          # release fraction, recovery time constant in ms

def x_ss(rate):
    d = np.exp(-1000 / (rate * tau_rec))       # recovery factor between spikes
    return ...

def train_amplitudes(rate, n=60):
    x, amps = 1.0, []
    for _ in range(n):
        amps.append(U * x)
        x = ...                                 # release, then recover until the next spike
    return np.array(amps)

r_half = brentq(lambda r: x_ss(r) - 0.5, 0.01, 100)
for r in [1, 5, 10, 20, 50]:
    print(f"{r:3d} Hz: steady amplitude {U * x_ss(r):.3f} (first {U:.2f}); amplitude times rate {U * x_ss(r) * r:.2f}")
print(f"the steady amplitude halves at {r_half:.2f} Hz")

In [ ]:
import numpy as np
def test_formula():
    "x_ss matches the steady state of the recursion for a regular train"
    x = 1.0
    for _ in range(200):
        x = x * (1 - U); x = 1 - (1 - x) * np.exp(-1000 / (20 * tau_rec))
    assert abs(x_ss(20) - x) < 1e-9
def test_sim():
    "the simulated last amplitude at 20 Hz agrees with U x_ss within 1%"
    assert abs(train_amplitudes(20)[-1] - U * x_ss(20)) < 0.01 * U
def test_half():
    "r_half is the rate at which the steady amplitude is half the first, and lies between 1 and 10 Hz"
    assert abs(x_ss(r_half) - 0.5) < 1e-6 and 1 < r_half < 10

test_formula()
test_sim()
test_half()
print("All checks passed")